# audio2chordpro（Google Colab）

音源 + 歌詞 → **歌詞の上にコードを載せた ChordPro** をブラウザの画面で作ります。

1. メニューの「ランタイム」→「ランタイムのタイプを変更」で **GPU（T4 など）** を選ぶ
2. 下のセルを実行し、出てきた **「audio2chordpro を開く」** をクリックする

あとは画面の中で、音源のアップロード → 曲情報・歌詞 → 作成 → コード譜のダウンロードまでできます。

- 初回はライブラリ（PyTorch など）とモデル（tsumugi、wav2vec2、Demucs、SheetSage2）のダウンロードに数分かかります
- 画面を使っている間は、このノートブックのタブを開いたままにしてください（閉じたり、しばらく操作しないとランタイムが止まります）
- 歌メロの採譜に使う [SheetSage2](https://huggingface.co/m-a-p/SheetSage2) の重みは **CC BY-NC 4.0（非商用）** です。
  使わない場合は、画面の「作成」→「詳しい設定」で歌メロを「MIDI の melody トラック」にしてください


In [ ]:
#@title ▶ audio2chordpro を起動する
#@markdown 実行すると、下に「audio2chordpro を開く」が出ます。
#@markdown
#@markdown プロジェクト（音源・解析結果・コード譜）を Google ドライブの `MyDrive/audio2chordpro` に保存して、次回も続きから使う：
USE_DRIVE = True  #@param {type:"boolean"}
REPO_URL = "https://github.com/anime-song/audio2chordpro"  #@param {type:"string"}

import socket
import subprocess
import time
from pathlib import Path

REPO = Path("/content/audio2chordpro")
PORT = 8000
LOG = Path("/content/audio2chordpro_server.log")


def running():
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", PORT)) == 0


if subprocess.run(["nvidia-smi"], capture_output=True).returncode != 0:
    print("⚠ GPU がありません。「ランタイム」→「ランタイムのタイプを変更」で GPU を選ぶと、解析がずっと速くなります")

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive/audio2chordpro")
else:
    DATA = Path("/content/audio2chordpro_data")
(DATA / "projects").mkdir(parents=True, exist_ok=True)

if not running():
    if not REPO.exists():
        print("▶ リポジトリを取得しています")
        !git clone -q --depth 1 {REPO_URL} {REPO}
    if not (REPO / ".venv" / ".ready").exists():
        print("▶ ライブラリをインストールしています（数分かかります）")
        !pip install -q uv
        !cd {REPO} && uv sync --frozen && touch .venv/.ready
    print("▶ 画面を用意しています")
    !cd {REPO} && uv run --frozen python -m audio2chordpro.server.web
    print("▶ サーバを起動しています")
    subprocess.Popen(
        ["uv", "run", "--frozen", "audio2chordpro", "serve", "--no-browser", "--port", str(PORT),
         "--root", str(DATA / "projects")],
        cwd=REPO, stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True,
    )
    for _ in range(180):
        if running():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"サーバが起動しませんでした。下の「サーバのログを見る」か {LOG} を確認してください")

from google.colab import output

print("✅ 準備できました。下のリンクから開いてください")
output.serve_kernel_port_as_window(PORT, anchor_text="▶ audio2chordpro を開く")


In [ ]:
#@title （困ったとき）サーバのログを見る
!tail -n 100 /content/audio2chordpro_server.log
